# NLP Assignment

Submission:
* There is no late submission for this assignment.
* It is individual work.
* Each student should submit an .ipynb file to the Teams Assignment.
* Grading: 0-15% of the course.

## Task
Here is a <a href="https://huggingface.co/datasets/banking77">dataset</a> composed of online banking queries annotated with their corresponding intents. You can download it by running the cell below. The data will be saved in the working directory.

DO NOT USE function ```datasets.load('banking77')```! Run the cell below and work with raw data.

What I expect you to do:
* Explore data: shape, number of classes, balance of classes. __2 points__.
* Solve a classification problem for a dataset using any <a href="https://huggingface.co/docs/transformers/tasks/sequence_classification">transformer</a> from the huggingface library. __3 points__.  
The tutorial at the link might be helpful.
* Justify choice of a metric. __3 points__.
* Split the training dataset into train and valid datasets. Train the model on train dataset and evaluate it on the valid dataset during training. Evaluate model on the test dataset after training. DO NOT USE a test dataset for validation!  __2 points__.
* Come up with 3 or more queries on banking topics and make a forecast of intents using your model. __2 points__.
* Comment code and describe your actions in the notebook. __1 point__.
* You must achieve a metric value of at least 90% __2 points__.
* Attach this file to the Teams Assignment. If you do not attach the file, you will get __0 points__.

The final score is calculated as a sum of all points.

If any of the tasks below will be completed with an error, the number of points for it may be reduced. For example, if you wrote only one query to the model instead of at least 3, then instead of __2 points__ you will get __1 point__.

## Notes
* Feel free to ask questions.
* Google Colab and Kaggle provide some free CPU and GPU time. Feel free to use it or use university resources.
* Good luck!

In [ ]:
%%bash
pip install transformers evaluate accelerate
wget -q -nc "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
wget -q -nc "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# Import pandas for data manipulation
import pandas as pd


In [ ]:
# Load train and test datasets from CSV files
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')


In [ ]:
# Preview the first rows of the training set
train_df.head()


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [ ]:
# Preview the first rows of the test set
test_df.head()


,text,category
0,How do I locate my card?,card_arrival
1,"I still have not received my new card, I order...",card_arrival
2,I ordered a card but it has not arrived. Help ...,card_arrival
3,Is there a way to know when my card will arrive?,card_arrival
4,My card has not arrived yet.,card_arrival


In [ ]:
# Check the shape of the training set
train_df.shape


(10003, 2)

In [ ]:
# Check the shape of the test set
test_df.shape


(3080, 2)

In [ ]:
# Inspect class distribution in the training set to assess balance
train_df['category'].value_counts()


,count
category,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
...,...
lost_or_stolen_card,82
card_swallowed,61
card_acceptance,59


In [ ]:
#Baseline - Logistic regression

In [ ]:
from sklearn.model_selection import train_test_split
X_train = train_df['text']
y_train = train_df['category']

# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


In [ ]:
X_test = test_df['text']
y_test = test_df['category']


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
clf = LogisticRegression(random_state=0, multi_class='ovr', solver='lbfgs', max_iter=2000)
clf.fit(X_train_tfidf, y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=2000, multi_class='ovr', random_state=0)

In [ ]:
# Predict on validation set
y_val_pred = clf.predict(X_val_tfidf)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_accuracy:.2f}")

# Predict on test set
y_test_pred = clf.predict(X_test_tfidf)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.2f}")

Validation Accuracy: 0.83
Test Accuracy: 0.83


In [ ]:
queries = [
    "How can I troubleshoot issues with card acceptance at ATMs and point-of-sale terminals?",
    "What should I do if my virtual card is not working for online transactions?",
    "How can I resolve problems with contactless payments not working on my card or mobile device?"
]

category = ["card_acceptance",
            "virtual_card_not_working",
            "contactless_not_working"]


In [ ]:
# Preprocess and vectorize the queries
queries_tfidf = vectorizer.transform(queries)

# Predict the category of the queries
predicted_categories = clf.predict(queries_tfidf)

# Print the predicted categories for each query
for i, query in enumerate(queries):
    print(f"Query: {query}")
    print(f"Predicted Category: {predicted_categories[i]}")
    print()

Query: How can I troubleshoot issues with card acceptance at ATMs and point-of-sale terminals?
Predicted Category: atm_support

Query: What should I do if my virtual card is not working for online transactions?
Predicted Category: card_not_working

Query: How can I resolve problems with contactless payments not working on my card or mobile device?
Predicted Category: card_not_working



In [ ]:
#Transformer

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset


In [ ]:
# Encode string category labels as integer IDs required by the model;
# store the encoder so we can map predictions back to label names later
label_encoder = LabelEncoder()
train_df['label'] = label_encoder.fit_transform(train_df['category'])
test_df['label'] = label_encoder.transform(test_df['category'])
num_classes = len(label_encoder.classes_)
num_classes


77

In [ ]:
# Split the training data into train (85%) and validation (15%) subsets
train_data, val_data = train_test_split(train_df, test_size=0.15, random_state=42)


In [ ]:
# Inspect the training split
train_data


,text,category,label
1746,"I have used all my PIN tries, what now?",pin_blocked,50
4693,I just noticed that I supposedly made a paymen...,direct_debit_payment_not_recognised,29
6876,I believe my card payment has reverted,reverted_card_payment?,53
4265,With my credit card i would like to transfer m...,topping_up_by_card,62
5918,My card did not work at store.,declined_card_payment,26
...,...,...,...
5734,It's been more than a week since I had a retur...,Refund_not_showing_up,0
5191,How long does it take for my payment to process?,pending_card_payment,46
5390,My statement has a charge that shouldn't have ...,request_refund,52
860,"I tried to withdraw money, but was unable to. ...",pending_cash_withdrawal,47


In [ ]:
# Load pre-trained DistilBERT tokenizer and fine-tuning model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_classes)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def preprocess_function(examples):
    """
    Tokenize text examples for the model.
    - truncation='longest_first': truncate from the end when the text exceeds max_length
    - padding='max_length': pad all sequences to max_length (512 tokens)
    - max_length=512: maximum sequence length accepted by DistilBERT
    """
    return tokenizer(examples['text'], truncation='longest_first', padding='max_length', max_length=512)

# Convert text data to token IDs and attention masks expected by the model
train_dataset = Dataset.from_pandas(train_data).map(preprocess_function, batched=True)
val_dataset   = Dataset.from_pandas(val_data).map(preprocess_function, batched=True)
test_dataset  = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)


Map:   0%|          | 0/8502 [00:00<?, ? examples/s]

Map:   0%|          | 0/1501 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [ ]:
# Compute accuracy and macro-F1 after each evaluation step
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    f1     = f1_score(labels, preds, average='macro')
    acc    = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1}


In [ ]:
# Configure training hyper-parameters:
# - 6 epochs with evaluation and checkpointing after every epoch
# - best checkpoint (highest macro-F1 on validation set) is reloaded at the end
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
    report_to='none'
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Initialise Trainer and run fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.211100,2.003881,0.674883,0.602041
2,1.036600,0.897922,0.838108,0.802745
3,0.680400,0.540409,0.891406,0.874483
4,0.404100,0.402673,0.910060,0.905469
5,0.302200,0.350534,0.921386,0.918073
6,0.213400,0.336722,0.920053,0.917563


TrainOutput(global_step=3192, training_loss=1.0561313076799077, metrics={'train_runtime': 2597.3312, 'train_samples_per_second': 19.64, 'train_steps_per_second': 1.229, 'total_flos': 6766465123971072.0, 'train_loss': 1.0561313076799077, 'epoch': 6.0})

In [ ]:
# Metric justification:
# - Accuracy: measures the percentage of correctly classified queries overall.
# - Macro-F1: harmonic mean of precision and recall averaged equally across all
#   77 classes; robust to class imbalance, which is present in this dataset.


In [ ]:
# Evaluate the fine-tuned model on the held-out test set
predictions = trainer.predict(test_dataset)
test_labels = test_df['label'].values
test_preds  = predictions.predictions.argmax(-1)
test_f1  = f1_score(test_labels, test_preds, average='macro')
test_acc = accuracy_score(test_labels, test_preds)
print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test F1-macro: {test_f1:.4f}')



Test Accuracy: 0.9088
Test F1-macro: 0.9088


In [ ]:
# Build a text-classification pipeline from the best fine-tuned model
from transformers import pipeline
classifier = pipeline('text-classification', model=trainer.model, tokenizer=tokenizer)


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [ ]:
# Run inference on custom banking queries.
# The pipeline returns 'LABEL_<id>' strings; we decode them back to
# the original category names using the fitted LabelEncoder.
new_texts = [
    'How can I troubleshoot issues with card acceptance at ATMs and point-of-sale terminals?',
    'What should I do if my virtual card is not working for online transactions?',
    'How can I resolve problems with contactless payments not working on my card or mobile device?'
]

for text in new_texts:
    result = classifier(text)[0]              # e.g. {'label': 'LABEL_4', 'score': 0.97}
    label_id = int(result['label'].split('_')[1])          # extract integer ID
    category = label_encoder.inverse_transform([label_id])[0]  # map to class name
    print(f'Query:     {text}')
    print(f'Predicted: {category}  (score: {result["score"]:.4f})')
    print()


[{'label': 'LABEL_4', 'score': 0.5033043622970581}]
[{'label': 'LABEL_41', 'score': 0.39055925607681274}]
[{'label': 'LABEL_24', 'score': 0.6511302590370178}]
